In [1]:
import re
import string
from collections import Counter
import numpy as np
import pandas as pd
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
from sklearn.metrics import classification_report, accuracy_score
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import Adam

nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Используется устройство: {device}")

Используется устройство: cuda


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\user\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\user\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\user\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [2]:
MAX_SEQUENCE_LENGTH = 25
EMBEDDING_DIM = 128
BATCH_SIZE = 64
NUM_EPOCHS = 25
LEARNING_RATE = 1e-4

In [3]:
df_train = pd.read_csv('./data/train.csv', encoding='1251')
df_test = pd.read_csv('./data/test.csv', encoding='1251')

df_train = df_train[['text', 'sentiment']].dropna()
df_test = df_test[['text', 'sentiment']].dropna()

df_train = df_train[df_train['sentiment'] != 'neutral']
df_test = df_test[df_test['sentiment'] != 'neutral']

print(df_train.shape, df_test.shape)

(16363, 2) (2104, 2)


In [4]:
stop_words = set(stopwords.words('english'))

def encode_target_label(label: str) -> int:
    if label == 'positive':
        return 1
    elif label == 'negative':
        return 0
    else:
        raise ValueError

def preprocess_text(text: str) -> str:
    text = text.lower()
    text = re.sub(r'\[.*?\]', '', text)
    text = re.sub(r"\\W", " ", text)
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    text = re.sub(r'<.*?>+', '', text)
    text = re.sub(rf'[{re.escape(string.punctuation)}]', '', text)
    text = re.sub(r'\n', ' ', text)
    text = re.sub(r'\w*\d\w*', '', text)
    
    words = word_tokenize(text)
    words = [word for word in words if word not in stop_words]
    words = [word for word in words if len(word) > 1]
    lemmatizer = WordNetLemmatizer()
    words = [lemmatizer.lemmatize(word) for word in words]
    
    return ' '.join(words)

df_train['sentiment'] = df_train['sentiment'].apply(encode_target_label)
df_test['sentiment'] = df_test['sentiment'].apply(encode_target_label)

df_train['text_transformed'] = df_train['text'].apply(preprocess_text)
df_test['text_transformed'] = df_test['text'].apply(preprocess_text)

df_train = df_train[df_train['text_transformed'].str.strip().astype(bool)]
df_test = df_test[df_test['text_transformed'].str.strip().astype(bool)]

In [5]:
full_corpus = pd.concat([df_train['text_transformed'], df_test['text_transformed']])

def build_vocab(corpus, min_freq=5):
    counter = Counter()
    for text in corpus:
        tokens = text.split()
        counter.update(tokens)
    vocab = {word for word, freq in counter.items() if freq >= min_freq}
    word2idx = {word: idx+2 for idx, word in enumerate(sorted(vocab))}
    word2idx['<PAD>'] = 0
    word2idx['<UNK>'] = 1
    idx2word = {idx: word for word, idx in word2idx.items()}
    return word2idx, idx2word

word2idx, idx2word = build_vocab(full_corpus, min_freq=5)
vocab_size = len(word2idx)
print(vocab_size)

3030


In [6]:
def text_to_sequence(text, word2idx):
    return [word2idx.get(word, word2idx['<UNK>']) for word in text.split()]

df_train['sequence'] = df_train['text_transformed'].apply(lambda x: text_to_sequence(x, word2idx))
df_test['sequence'] = df_test['text_transformed'].apply(lambda x: text_to_sequence(x, word2idx))

In [7]:
class TextDataset(Dataset):
    def __init__(self, sequences, labels):
        self.sequences = sequences
        self.labels = labels
    
    def __len__(self):
        return len(self.sequences)
    
    def __getitem__(self, idx):
        return self.sequences[idx], self.labels[idx]

train_dataset = TextDataset(df_train['sequence'].tolist(), df_train['sentiment'].tolist())
test_dataset = TextDataset(df_test['sequence'].tolist(), df_test['sentiment'].tolist())

def collate_fn(batch):
    sequences, labels = zip(*batch)
    sequences_padded = []
    for seq in sequences:
        if len(seq) > MAX_SEQUENCE_LENGTH:
            sequences_padded.append(seq[:MAX_SEQUENCE_LENGTH])
        else:
            sequences_padded.append(seq + [word2idx['<PAD>']] * (MAX_SEQUENCE_LENGTH - len(seq)))
    sequences_padded = torch.tensor(sequences_padded, dtype=torch.long)
    labels = torch.tensor(labels, dtype=torch.float)
    return sequences_padded, labels

train_loader = DataLoader(dataset=train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
test_loader = DataLoader(dataset=test_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

In [8]:
class CNN(nn.Module):
    def __init__(self, vocab_size, embedding_dim, num_classes=1, dropout=0.5):
        super(CNN, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=word2idx['<PAD>'])
        self.conv1 = nn.Conv1d(in_channels=embedding_dim, out_channels=128, kernel_size=5)
        self.relu = nn.ReLU()
        self.pool = nn.MaxPool1d(kernel_size=2)
        conv_output_length = (MAX_SEQUENCE_LENGTH - 5 + 1) // 2
        self.fc1 = nn.Linear(128 * conv_output_length, 64)
        self.dropout = nn.Dropout(dropout)
        self.fc2 = nn.Linear(64, num_classes)
        self.sigmoid = nn.Sigmoid()
    
    def forward(self, x):
        x = self.embedding(x)
        x = x.permute(0, 2, 1)
        x = self.conv1(x)
        x = self.relu(x)
        x = self.pool(x)
        x = x.view(x.size(0), -1)
        x = self.fc1(x)
        x = self.relu(x)
        x = self.dropout(x)
        x = self.fc2(x)
        x = self.sigmoid(x)
        return x.squeeze()

In [9]:
def train_model(model, train_loader, criterion, optimizer, epochs=NUM_EPOCHS):
    for epoch in range(1, epochs + 1):
        model.train()
        train_losses = []
        for sequences, labels in train_loader:
            sequences = sequences.to(device)
            labels = labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(sequences)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            train_losses.append(loss.item())
        
        avg_train_loss = np.mean(train_losses)
        print(f"Epoch {epoch}/{epochs} | Loss: {avg_train_loss:.4f}")

def evaluate_model(model, data_loader):
    model.eval()
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for sequences, labels in data_loader:
            sequences = sequences.to(device)
            outputs = model(sequences)
            preds = (outputs > 0.5).float()
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.numpy())
    return all_preds, all_labels

In [10]:
model = CNN(vocab_size=vocab_size, embedding_dim=EMBEDDING_DIM).to(device)
criterion = nn.BCELoss()
optimizer = Adam(model.parameters(), lr=LEARNING_RATE)

In [11]:
train_model(model, train_loader, criterion, optimizer, epochs=NUM_EPOCHS)

Epoch 1/25 | Loss: 0.6765
Epoch 2/25 | Loss: 0.5791
Epoch 3/25 | Loss: 0.4854
Epoch 4/25 | Loss: 0.4225
Epoch 5/25 | Loss: 0.3702
Epoch 6/25 | Loss: 0.3237
Epoch 7/25 | Loss: 0.2784
Epoch 8/25 | Loss: 0.2352
Epoch 9/25 | Loss: 0.1965
Epoch 10/25 | Loss: 0.1619
Epoch 11/25 | Loss: 0.1344
Epoch 12/25 | Loss: 0.1074
Epoch 13/25 | Loss: 0.0857
Epoch 14/25 | Loss: 0.0702
Epoch 15/25 | Loss: 0.0583
Epoch 16/25 | Loss: 0.0472
Epoch 17/25 | Loss: 0.0383
Epoch 18/25 | Loss: 0.0324
Epoch 19/25 | Loss: 0.0284
Epoch 20/25 | Loss: 0.0249
Epoch 21/25 | Loss: 0.0229
Epoch 22/25 | Loss: 0.0198
Epoch 23/25 | Loss: 0.0177
Epoch 24/25 | Loss: 0.0158
Epoch 25/25 | Loss: 0.0150


In [12]:
y_pred_train, y_true_train = evaluate_model(model, train_loader)
y_pred_test, y_true_test = evaluate_model(model, test_loader)
print(f'Train accuracy: {accuracy_score(y_true_train, y_pred_train):.4f}')
print(classification_report(y_true_train, y_pred_train, target_names=['negative', 'positive']))
print(f'Test accuracy: {accuracy_score(y_true_test, y_pred_test):.4f}')
print(classification_report(y_true_test, y_pred_test, target_names=['negative', 'positive']))

Train accuracy: 0.9968
              precision    recall  f1-score   support

    negative       1.00      1.00      1.00      7777
    positive       1.00      1.00      1.00      8580

    accuracy                           1.00     16357
   macro avg       1.00      1.00      1.00     16357
weighted avg       1.00      1.00      1.00     16357

Test accuracy: 0.8169
              precision    recall  f1-score   support

    negative       0.79      0.84      0.81      1000
    positive       0.85      0.80      0.82      1103

    accuracy                           0.82      2103
   macro avg       0.82      0.82      0.82      2103
weighted avg       0.82      0.82      0.82      2103



In [17]:
for i in range(6):
    print(f'Original sentence: {df_test.iloc[i]["text"]}')
    print(f'Transformed sentence: {df_test.iloc[i]["text_transformed"]}')
    print(f'Actual label: {df_test.iloc[i]["sentiment"]}')
    print(f'Predicted label: {y_pred_test[i]}')
    print('-' * 50)

Original sentence:  Shanghai is also really exciting (precisely -- skyscrapers galore). Good tweeps in China:  (SH)  (BJ).
Transformed sentence: shanghai also really exciting precisely skyscraper galore good tweeps china sh bj
Actual label: 1
Predicted label: 1.0
--------------------------------------------------
Original sentence: Recession hit Veronique Branquinho, she has to quit her company, such a shame!
Transformed sentence: recession hit veronique branquinho quit company shame
Actual label: 0
Predicted label: 0.0
--------------------------------------------------
Original sentence:  happy bday!
Transformed sentence: happy bday
Actual label: 1
Predicted label: 1.0
--------------------------------------------------
Original sentence:  http://twitpic.com/4w75p - I like it!!
Transformed sentence: like
Actual label: 1
Predicted label: 1.0
--------------------------------------------------
Original sentence:  that`s great!! weee!! visitors!
Transformed sentence: thats great weee visit